# RL Attack Path Simulation -- Cloud Training (Colab)
## MMAI 845 | Syed Ali Turab

**Instructions:**
1. Open this notebook in Google Colab
2. Go to Runtime > Change runtime type > Select **T4 GPU**
3. Run all cells (Runtime > Run all)
4. Download the `results/` folder when done

Training 500k steps per agent takes ~10-20 minutes on a T4 GPU.

**Key features:**
- MaskablePPO (sb3-contrib) with NASim action masking
- DQN with manual Q-value masking for invalid actions
- Dense reward shaping for exploration
- Fully observable environment

---
## 1. Setup

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Clone the repository
!git clone https://github.com/turaab97/rl-attack-path-simulation.git
%cd rl-attack-path-simulation

In [ ]:
# Install dependencies (includes sb3-contrib for MaskablePPO)
!pip install -e ".[dev]" -q

In [ ]:
# Verify environment and action masking
from environments.network_config import make_env, AI_INFRA_HOSTS
from agents.wrappers import IntActionWrapper, ActionMaskWrapper, DenseRewardWrapper
import numpy as np

env = make_env()
obs, info = env.reset()
print(f'Environment OK: obs={obs.shape}, actions={env.action_space.n}')
print(f'AI Infrastructure targets: {AI_INFRA_HOSTS}')
print(f'Fully observable: obs has {np.count_nonzero(obs)}/{obs.shape[0]} non-zero features')

# Verify action mask wrapper
masked_env = ActionMaskWrapper(IntActionWrapper(env))
masked_env.reset()
mask = masked_env.action_masks()
n_valid = int(mask.sum())
print(f'Action masking: {n_valid}/{env.action_space.n} valid actions from initial state')

# Verify dense reward wrapper
wrapped = DenseRewardWrapper(ActionMaskWrapper(IntActionWrapper(make_env())))
obs2, _ = wrapped.reset()
_, r1, _, _, _ = wrapped.step(0)
print(f'Dense reward wrapper OK (noop reward: {r1:.2f})')
wrapped.close()
masked_env.close()
print('All checks passed.')

---
## 2. Baseline Training (PPO + DQN, 500k steps)

In [ ]:
!python -m training.train --compare --timesteps 500000 --eval_freq 25000 --n_eval_episodes 10 --seed 42

---
## 3. Stealth Training (PPO + DQN, 500k steps)

In [ ]:
!python -m training.train --compare --stealth --timesteps 500000 \
    --detection_threshold 0.8 --detection_cost 0.1 --caught_penalty -100.0 --alpha 1.0 \
    --eval_freq 25000 --n_eval_episodes 10 --seed 42

---
## 4. Evaluation

In [ ]:
# Evaluate baseline models
!python -m training.evaluate \
    --ppo_model results/ppo_baseline/final_model \
    --dqn_model results/dqn_baseline/final_model \
    --episodes 100

In [ ]:
# Evaluate stealth models
!python -m training.evaluate \
    --ppo_model results/ppo_stealth/final_model \
    --dqn_model results/dqn_stealth/final_model \
    --stealth --episodes 100

---
## 5. Generate Plots

In [ ]:
!python -m analysis.visualize --results_dir results/ --output_dir results/plots

---
## 6. Generate Pentest Report

In [ ]:
!python -m analysis.report_generator --results_dir results/ --output results/pentest_report.md

---
## 7. Verify Output

In [ ]:
import json
from pathlib import Path

results_dir = Path('results')

print('=== Training Artifacts ===')
for f in sorted(results_dir.rglob('*')):
    if f.is_file():
        size = f.stat().st_size
        unit = 'KB' if size > 1024 else 'B'
        val = size / 1024 if size > 1024 else size
        print(f'  {f.relative_to(results_dir)}  ({val:.1f} {unit})')

eval_path = results_dir / 'eval_results.json'
if eval_path.exists():
    print()
    print('=== Evaluation Results ===')
    with open(eval_path) as f:
        data = json.load(f)
    for agent, metrics in data.items():
        print(f'  {agent.upper()}:')
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f'    {k}: {v:.4f}')
else:
    print('No eval_results.json found.')

for meta_path in sorted(results_dir.rglob('train_meta.json')):
    print(f'\n=== {meta_path.parent.name} Training Metadata ===')
    with open(meta_path) as f:
        meta = json.load(f)
    print(f'  Steps: {meta.get("total_timesteps", "?")}')
    print(f'  Wall time: {meta.get("wall_time_seconds", "?")}s')
    print(f'  Seed: {meta.get("seed", "?")}')

---
## 8. Download Results

Download the full `results/` directory to your local machine,
then copy it into your local repo clone to use with the analysis notebook.

In [ ]:
# Zip results for download
!zip -r /content/rl_results.zip results/

from google.colab import files
files.download('/content/rl_results.zip')
print('\nDownload started. Copy the results/ folder into your local repo clone.')

---

*Training notebook by Syed Ali Turab -- MMAI 845, Queen's University*